# RAG CGU: fluxo completo, local (do banco à avaliação, com métricas e gráficos)
Execute as células **em ordem**. Tudo roda na sua máquina: banco SQLite, ingestão, limpeza, enriquecimento, embeddings, busca, LLM local, avaliação e gráficos de progresso.
A lógica está em `src/cgu_rag/`; aqui você dispara, mede e inspeciona. Cada medição é gravada em `results/progresso.json` e os gráficos em `results/*.png` (evidência para o README).

**Após reiniciar o kernel:** rode só as células 0, 1 (as duas) e **1b (Retomada)**; depois vá direto à etapa desejada. Banco e embeddings ficam em disco.

**Escopo:** este notebook é a fase experimental (baseline de recuperação, corpus, latência). O agente da Unidade II (LangGraph, MCP, skills, guardrails, Streamlit) vive em `src/` e `app/`; veja a etapa 11.

## 0. Ambiente
Localiza a raiz do projeto, ajusta o diretório de trabalho e confere as dependências.

In [ ]:
import os, sys, importlib.util
from pathlib import Path

# raiz do projeto = pasta que contém pyproject.toml (funciona se o notebook estiver em notebooks/ ou na raiz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))          # permite rodar sem "pip install -e ."
print("Raiz:", RAIZ, "| Python:", sys.version.split()[0])

faltando = [m for m in ("fitz", "numpy", "requests", "sentence_transformers", "openai") if importlib.util.find_spec(m) is None]
if faltando:
    print("FALTANDO:", faltando)
    print('Instale com:  pip install -e ".[notebook]"   (no terminal, dentro do ambiente virtual) e reinicie o kernel')
else:
    print("Dependências OK")

## 1. Configuração
Caminhos, modelo de embeddings, LLM local e política de sigilo. Edite só esta célula.

In [ ]:
import os

os.environ["RAG_DB"] = "data/rag_cgu.db"                          # arquivo do banco (criado na etapa 2)
os.environ["RAG_EMB_MODEL"] = "intfloat/multilingual-e5-small"    # baseline; troque só depois do benchmark
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"          # Ollama local
os.environ["LLM_MODEL"] = "jurema-7b"                             # nome criado no "ollama create"
os.environ["SIGILO"] = "sigiloso"                                 # "sigiloso" = só endpoint local (falha fechada)

PASTA_PDFS = "data/pdfs"
GABARITO = "data/gabarito/perguntas.jsonl"

In [ ]:
%load_ext autoreload
%autoreload 2
from cgu_rag import config
from cgu_rag.legado import rag_cgu_pipeline as legado
from cgu_rag.busca import buscar, responder
print(config.settings)

## 1b. Retomada (idempotente)
Recria `con`, `llm`, `mostrar` e os imports. Rode depois de reiniciar o kernel; é inofensiva se rodada de novo.

In [ ]:
from pathlib import Path
from cgu_rag.llm_local import LLMLocal

Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)
con = legado.connect()
llm = LLMLocal()          # usa LLM_MODEL=jurema-7b da etapa 1

def mostrar(q, k=5, denso=True):
    for score, r in buscar(con, q, k, denso):
        print(f"[{score:.4f}] {(r['nota_tecnica'] or r['doc_titulo'])[:55]} | {r['titulo'][:70]} (págs {r['pag_ini']}-{r['pag_fim']})")

print("trechos:", con.execute("SELECT COUNT(*) FROM chunks WHERE tipo!='indice'").fetchone()[0],
      "| com vetor:", con.execute("SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL").fetchone()[0])

# --- ferramentas de métricas e gráficos (uma vez por kernel) ---
import json, time, subprocess, importlib.util
if any(importlib.util.find_spec(m) is None for m in ("matplotlib", "pandas")):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "pandas"])
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

RES = Path("results"); RES.mkdir(exist_ok=True)
PROG = RES / "progresso.json"

def registrar(etapa, experimento, metricas):
    """Grava/atualiza uma medição em results/progresso.json (idempotente por nome de experimento)."""
    h = json.loads(PROG.read_text(encoding="utf-8")) if PROG.exists() else []
    h = [x for x in h if x["experimento"] != experimento]
    h.append({"etapa": etapa, "experimento": experimento, "quando": time.strftime("%Y-%m-%d %H:%M"), "metricas": metricas})
    PROG.write_text(json.dumps(h, ensure_ascii=False, indent=2), encoding="utf-8")

def salvar_fig(nome):
    plt.tight_layout(); plt.savefig(RES / f"{nome}.png", dpi=130); plt.show()

## 2. Criar o banco
`legado.connect()` cria o arquivo SQLite e todo o esquema (`documents`, `chunks`, índice de texto `chunks_fts` com BM25, `meta`).
É idempotente: se o banco já existe, só abre (e migra colunas antigas).

In [ ]:
from pathlib import Path
Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)

con = legado.connect()
print("Banco:", Path(legado.DB_PATH).resolve())
for (nome, tipo) in con.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table','trigger') ORDER BY type, name"):
    print(f"  {tipo:8s} {nome}")

In [ ]:
# OPCIONAL: recomeçar do zero (apaga o banco). Descomente para usar.
# con.close(); Path(legado.DB_PATH).unlink(missing_ok=True); con = legado.connect(); print("banco recriado")

## 3. Ingestão dos PDFs
Extrai o texto com PyMuPDF, limpa, segmenta por seções e grava os trechos. PDFs já ingeridos e sem mudança são ignorados (hash SHA-256).

In [ ]:
import os, re, time, hashlib
from pathlib import Path
import requests

# ── Configuração ─────────────────────────────────────────────────────────────
os.environ["RAG_DB"] = "data/rag_cgu.db"
os.environ["RAG_EMB_MODEL"] = "intfloat/multilingual-e5-small"
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODEL"] = "jurema-7b"
os.environ["SIGILO"] = "sigiloso"

FONTES = [
    "https://basedeconhecimento.cgu.gov.br/bitstreams/ec9c2e26-8443-4cae-9d65-7abea5076ca3/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/f1ebf513-6f49-489e-9220-7cfd12b08077/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/bcd1fc60-09d0-488a-936b-ab1749ecbfed/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/f0476e82-972f-47a0-b7a1-9b4c4d12471f/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/0d1ed7f8-6300-4e07-b596-d1d344a1726f/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/d699bba4-e046-4e06-ae38-8fdc1e148e60/download",
]
PASTA_PDFS = "data/pdfs"          # se o PDF já estiver aqui, não baixa; senão baixa e salva aqui
GABARITO = "data/gabarito/perguntas.jsonl"

# ── Garante legado e banco (caso as etapas 1 e 2 ainda não tenham sido rodadas) ──
from cgu_rag import config
from cgu_rag.legado import rag_cgu_pipeline as legado
Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)
try:
    con
except NameError:
    con = legado.connect()
Path(PASTA_PDFS).mkdir(parents=True, exist_ok=True)

# ── Download ou pasta ────────────────────────────────────────────────────────
def arquivo_da_fonte(src: str) -> Path:
    m = re.search(r"/bitstreams/([0-9a-fA-F-]{36})/", src)
    nome = m.group(1) if m else hashlib.sha1(src.encode()).hexdigest()[:12]
    return Path(PASTA_PDFS) / f"{nome}.pdf"

def fetch_pasta_ou_download(src: str) -> bytes:
    if not re.match(r"https?://", src):
        return Path(src).read_bytes()
    arq = arquivo_da_fonte(src)
    if arq.exists() and arq.stat().st_size > 0:
        print(f"  na pasta, sem baixar: {arq.name}")
        return arq.read_bytes()
    ultimo = None
    for tentativa in range(1, 4):
        try:
            r = requests.get(src, timeout=180, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)
            r.raise_for_status()
            arq.write_bytes(r.content)
            print(f"  baixado {len(r.content)/1e6:.1f} MB -> {arq.name}")
            return r.content
        except Exception as e:
            ultimo = e
            time.sleep(2 * tentativa)
    raise RuntimeError(f"falha ao baixar {src}: {ultimo}")

legado.fetch = fetch_pasta_ou_download

nomes_dos_links = {arquivo_da_fonte(u).name for u in FONTES}
locais = [str(p) for p in sorted(Path(PASTA_PDFS).glob("*.pdf")) if p.name not in nomes_dos_links]
fontes = list(dict.fromkeys(FONTES + locais))
print(len(fontes), "fonte(s):", len(FONTES), "link(s) +", len(locais), "PDF(s) local(is) extra(s)")
assert fontes, "Defina links em FONTES ou coloque PDFs em data/pdfs/"

# ── Ingestão ─────────────────────────────────────────────────────────────────
for src in fontes:
    try:
        legado.ingest(con, src)
    except Exception as e:
        print(f"! erro em {src[-60:]}: {e}")
con.commit()

In [ ]:
for r in con.execute("SELECT id, titulo, n_paginas FROM documents"):
    c = con.execute("SELECT COUNT(*) n, SUM(tipo='secao') s FROM chunks WHERE doc_id=? AND tipo!='indice'", (r["id"],)).fetchone()
    print(f"doc {r['id']}: {r['titulo'][:60]!r} | {r['n_paginas']} págs | {c['n']} trechos | {c['s']} seções")

## 4. Relatório de qualidade da limpeza
Cobertura de páginas, tamanho dos trechos e ruído residual (cabeçalho/rodapé, hífens, caracteres de controle).

In [ ]:
for r in legado.quality_report(con):
    print(f"\ndoc {r['doc']}: {r['titulo']!r} | {r['paginas']} págs ({r['paginas_cobertas']} cobertas) | {r['chunks']} trechos "
          f"(mediana {r['tam_mediano']} car., {r['curtos_300']} < 300)")
    print("  ruído residual:", {k: v for k, v in r["ruido"].items()})

## 4b. Panorama do corpus (gráficos)
Mostra o tamanho do corpus e a distribuição dos trechos. Serve para justificar a segmentação (decisão de projeto) e fica gravado como medição.

In [ ]:
df = pd.read_sql_query("""
    SELECT c.rowid AS cid, d.id AS doc, substr(d.titulo,1,38) AS doc_titulo, d.n_paginas, c.tipo,
           length(c.texto) AS n_car
    FROM chunks c JOIN documents d ON d.id = c.doc_id WHERE c.tipo != 'indice'""", con)
docs = df.groupby(["doc", "doc_titulo"]).agg(trechos=("cid", "count"), paginas=("n_paginas", "first")).reset_index()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].barh(docs["doc"].astype(str) + " " + docs["doc_titulo"], docs["trechos"]); ax[0].set_title("Trechos por documento")
ax[1].barh(docs["doc"].astype(str) + " " + docs["doc_titulo"], docs["paginas"], color="tab:orange"); ax[1].set_title("Páginas por documento")
ax[2].hist(df["n_car"].clip(upper=6000), bins=40); ax[2].axvline(df["n_car"].median(), color="r", ls="--")
ax[2].set_title("Tamanho dos trechos (caracteres)")
salvar_fig("01_corpus")

m = {"documentos": int(docs.shape[0]), "trechos": int(df.shape[0]), "paginas": int(docs["paginas"].sum()),
     "mediana_car": int(df["n_car"].median()), "trechos_curtos_300": int((df["n_car"] < 300).sum())}
print(m); registrar("ingestão", "corpus (6 PDFs CGU)", m)

## 5. Enriquecimento (heurístico, sem LLM)
Gera resumo, palavras-chave, perguntas e contexto por regras. O enriquecimento **por LLM** do módulo legado usa provedores externos,
por isso não é usado aqui enquanto `SIGILO=sigiloso`.

In [ ]:
legado.enrich_pending(con, "heuristic")
con.commit()

## 6. Embeddings
Primeira execução baixa o modelo de embeddings (precisa de internet uma vez); depois fica em cache local. O banco trava o modelo usado: para trocar de modelo, limpe os embeddings.

In [ ]:
import time
t0 = time.time()
legado.embed_pending(con)
print(f"{time.time()-t0:.0f} s | modelo:", con.execute("SELECT v FROM meta WHERE k='emb_model'").fetchone()["v"])

In [ ]:
print("trechos:", con.execute("SELECT COUNT(*) FROM chunks WHERE tipo!='indice'").fetchone()[0],
      "| com vetor:", con.execute("SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL").fetchone()[0])

## 7. Busca (BM25 + vetores, fusão RRF)

In [ ]:
def mostrar(q, k=5, denso=True):
    for score, r in buscar(con, q, k, denso):
        print(f"[{score:.4f}] {(r['nota_tecnica'] or r['doc_titulo'])[:55]} | {r['titulo'][:70]} (págs {r['pag_ini']}-{r['pag_fim']})")

mostrar("prazo para instauração de PAD")

Comparação rápida: só BM25 contra BM25 + vetores (use perguntas suas):

In [ ]:
q = "prazo para instauração de PAD"
print("--- só BM25"); mostrar(q, 5, denso=False)
print("--- BM25 + vetores"); mostrar(q, 5, denso=True)

## 7b. Lei de Zipf no corpus
Frequência x posição das palavras em escala log-log. Uma reta de inclinação próxima de -1 confirma a cauda longa típica de texto jurídico e justifica usar BM25/TF-IDF (ponderação por raridade) e remover palavras vazias.

In [ ]:
import re
from collections import Counter

cont = Counter()
for (t,) in con.execute("SELECT texto FROM chunks WHERE tipo != 'indice'"):
    cont.update(re.findall(r"[a-záéíóúâêôãõçà]{2,}", t.lower()))
freq = np.array(sorted(cont.values(), reverse=True), dtype=float)
rank = np.arange(1, len(freq) + 1)
a = slice(0, min(5000, len(freq)))               # ajuste no trecho central da curva
slope = np.polyfit(np.log(rank[a]), np.log(freq[a]), 1)[0]

plt.figure(figsize=(6, 4.5))
plt.loglog(rank, freq, ".", ms=3, label="corpus")
plt.loglog(rank, freq[0] / rank, "r--", lw=1, label="Zipf ideal (1/r)")
plt.xlabel("posição (rank)"); plt.ylabel("frequência"); plt.title(f"Lei de Zipf (inclinação {slope:.2f})"); plt.legend()
salvar_fig("02_zipf")
print("top 15:", [w for w, _ in cont.most_common(15)])
registrar("nlp", "zipf", {"vocabulario": int(len(freq)), "tokens": int(freq.sum()), "inclinacao": round(float(slope), 3)})

## 8. Testar o LLM local
Se falhar, o servidor não está rodando ou o nome do modelo difere do criado no Ollama.

In [ ]:
from cgu_rag.llm_local import LLMLocal
llm = LLMLocal()
try:
    print(llm.chat([{"role": "user", "content": "Responda apenas: pronto."}], max_tokens=10))
    print(llm.medir("Explique em duas frases o que é processo administrativo disciplinar.", max_tokens=64))
except Exception as e:
    print("LLM indisponível:", repr(e))
    print("Confira: 'ollama list' mostra", config.settings.llm_model, "e o servidor está em", config.settings.llm_base_url)

## 8b. Latência com contexto de RAG (~3000 tokens)
Mede o tempo até o 1º token e a velocidade com um prompt do tamanho de uma resposta de RAG. É o número que decide quantos trechos (k) e qual modelo usar no agente.

In [ ]:
BASE = ("A Controladoria-Geral da União realiza atividades de correição para apurar responsabilidades "
        "administrativas, conforme as normas aplicáveis ao processo administrativo disciplinar. ")
prompt = (BASE * (12000 // len(BASE) + 1))[:12000] + "\n\nResuma o texto acima em duas frases."
llm.medir("Diga olá.", max_tokens=8)             # aquecimento
r = llm.medir(prompt, max_tokens=64); print(r)
registrar("llm", f"latência {config.settings.llm_model} ~3000 tokens", r)

## 9. Resposta com citação
Busca os trechos e pede ao LLM uma resposta **só com base neles**, citando documento e páginas.

In [ ]:
pergunta = "Quando a prescrição da pretensão punitiva começa a correr?"
out = responder(con, pergunta, k=4, llm=llm)
print(out["resposta"]); print()
for f in out["fontes"]: print("-", f)

## 10. Avaliação da recuperação (hit@k, MRR) e progresso
**Gabarito:** se `data/gabarito/perguntas.jsonl` não existir (ou ainda tiver o texto-modelo), gera-se um gabarito **sintético**: para 30 seções sorteadas (semente fixa), o LLM local escreve uma pergunta que o trecho responde. O acerto é **estrito**: mesmo documento, mesmo título de seção e páginas sobrepostas.
**Limite conhecido:** perguntas sintéticas favorecem a busca lexical (o LLM tende a reutilizar termos do trecho). Revise as perguntas e, quando possível, substitua pelo benchmark original (formato: `id`, `pergunta`, `esperado`).

In [ ]:
import random
g = Path(GABARITO); g.parent.mkdir(parents=True, exist_ok=True)
N = 30
if (not g.exists()) or "TROQUE-PELA" in g.read_text(encoding="utf-8"):
    rows = con.execute("""SELECT d.titulo AS doc, c.titulo, c.pag_ini, c.pag_fim, c.texto
        FROM chunks c JOIN documents d ON d.id = c.doc_id
        WHERE c.tipo = 'secao' AND length(c.texto) BETWEEN 600 AND 2500""").fetchall()
    random.seed(42); amostra = random.sample(rows, min(N, len(rows))); itens = []
    for i, r in enumerate(amostra, 1):
        msg = [{"role": "system", "content": "Você cria perguntas de avaliação. Responda SOMENTE com uma pergunta objetiva em português que o trecho responde. Não cite o título nem números de página."},
               {"role": "user", "content": r["texto"][:1200]}]
        q = llm.chat(msg, max_tokens=60, temperature=0.3).strip().split("\n")[0]
        itens.append({"id": f"q{i:02d}", "pergunta": q, "esperado": r["titulo"], "doc": r["doc"],
                      "pag_ini": r["pag_ini"], "pag_fim": r["pag_fim"], "origem": "sintetico"})
    g.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in itens), encoding="utf-8")
    print(len(itens), "perguntas geradas em", g)
else:
    print("gabarito existente mantido:", g)
itens = [json.loads(l) for l in g.read_text(encoding="utf-8").splitlines() if l.strip()]
pd.DataFrame(itens)[["id", "pergunta", "esperado"]].head(8)

In [ ]:
from cgu_rag.avaliacao import acertou
KS = (1, 3, 5, 10)

def acerto(it, r):
    if "doc" in it:   # estrito: mesmo doc + mesma seção + páginas sobrepostas
        return (r["doc_titulo"] == it["doc"] and r["titulo"] == it["esperado"]
                and r["pag_ini"] <= it["pag_fim"] and r["pag_fim"] >= it["pag_ini"])
    return acertou(it["esperado"], r)   # gabarito original (frouxo)

def avaliar(itens, denso):
    pos, t0 = [], time.time()
    for it in itens:
        linhas = [r for _, r in buscar(con, it["pergunta"], max(KS), denso)]
        pos.append(next((i for i, r in enumerate(linhas, 1) if acerto(it, r)), None))
    n = len(pos)
    m = {f"hit@{k}": round(sum(1 for p in pos if p and p <= k) / n, 4) for k in KS}
    m.update({"mrr@10": round(sum(1 / p for p in pos if p) / n, 4), "n": n,
              "s_por_consulta": round((time.time() - t0) / n, 3)})
    return m, pos

### 10.1 Baseline: só BM25 x BM25 + vetores (RRF)

In [ ]:
m_bm25, pos_bm25 = avaliar(itens, denso=False)
m_hib, pos_hib = avaliar(itens, denso=True)
registrar("baseline", "A1 BM25", m_bm25); registrar("baseline", "A2 BM25+vetores (RRF)", m_hib)

tab = pd.DataFrame({"BM25": m_bm25, "BM25+vetores": m_hib}).T
print(tab.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(KS)); w = 0.38
ax[0].bar(x - w/2, [m_bm25[f"hit@{k}"] for k in KS], w, label="BM25")
ax[0].bar(x + w/2, [m_hib[f"hit@{k}"] for k in KS], w, label="BM25+vetores")
ax[0].set_xticks(x); ax[0].set_xticklabels([f"hit@{k}" for k in KS]); ax[0].set_ylim(0, 1); ax[0].legend(); ax[0].set_title("Recuperação por k")
maxp = 11
ax[1].hist([[p or maxp for p in pos_bm25], [p or maxp for p in pos_hib]], bins=range(1, maxp + 2), label=["BM25", "BM25+vetores"])
ax[1].set_title("Posição do acerto (11 = não achou)"); ax[1].legend()
salvar_fig("03_baseline")

### 10.2 Análise de erros (perguntas não recuperadas no top-5 pelo modelo híbrido)

In [ ]:
falhas = [(it, p) for it, p in zip(itens, pos_hib) if not p or p > 5]
print(f"{len(falhas)} de {len(itens)} fora do top-5")
for it, p in falhas[:8]:
    print(f"- {it['id']} (posição {p}): {it['pergunta']}\n    esperado: {it['esperado'][:80]}")

### 10.3 Progresso entre experimentos
Cada linha é uma medição registrada. Toda mudança futura (novo embedding, reranker, metadados do LegalNLP, agente) deve ser comparada com o baseline A1/A2.

In [ ]:
h = json.loads(PROG.read_text(encoding="utf-8"))
ret = [x for x in h if "hit@5" in x["metricas"]]
if ret:
    nomes = [x["experimento"] for x in ret]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(nomes, [x["metricas"]["hit@5"] for x in ret], "o-", label="hit@5")
    ax.plot(nomes, [x["metricas"]["mrr@10"] for x in ret], "s--", label="MRR@10")
    ax.set_ylim(0, 1); ax.legend(); ax.set_title("Progresso da recuperação"); plt.xticks(rotation=20, ha="right")
    salvar_fig("04_progresso")
print(pd.DataFrame([{"etapa": x["etapa"], "experimento": x["experimento"], "quando": x["quando"], **x["metricas"]} for x in h]).to_string())

## 11. Rastreabilidade com a Avaliação da Unidade II
Este notebook cobre a **base** (corpus, recuperação, LLM local, métricas). O restante é construído em `src/` e `app/` e documentado no README (relatório).

| # | Requisito | Onde | Status |
|---|---|---|---|
| 1 | Grafo LangGraph (estado, nós, arestas próprios) | `src/cgu_rag/agente/grafo.py` | pendente |
| 2 | RAG agêntico (decide consultar, reformula, rebusca; informa a origem) | nó `recuperar` + `buscar()` deste notebook | base pronta |
| 3 | MCP: 1 servidor próprio + 1 da comunidade | `mcp_servers/cgu_normas.py` + a definir | pendente |
| 4 | 2 skills em `SKILL.md`, carregadas sob demanda | `skills/*/SKILL.md` | pendente |
| 5 | Memória com checkpointer (thread por conversa) | `grafo.py` | pendente |
| 6 | Guardrails de entrada e saída (nós) | `guardrails.py` | pendente |
| 7 | Interface Streamlit (chat, nova conversa, ferramentas e skills por resposta) | `app/streamlit_app.py` | pendente |
| 8 | README-relatório + 10 casos de teste + vídeo de 5 min | `README.md` | pendente |

**Decisões já tomadas:** domínio = correição e processo disciplinar da CGU; documentos = 6 PDFs públicos da Base de Conhecimento da CGU (links na etapa 3); LLM local Jurema-7B (Ollama) com política de sigilo; baseline = BM25 + vetores com RRF.